# Project 3

In [1]:
import os
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("Project3-CDC")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", os.environ.get("AWS_ACCESS_KEY_ID", "admin"))
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ.get("AWS_SECRET_ACCESS_KEY", "password"))
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Extensions:", spark.conf.get("spark.sql.extensions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

Spark version: 4.1.0
Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
AQE enabled: false


In [35]:
# Bronze CDC table
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")

spark.sql('''
CREATE TABLE IF NOT EXISTS lakehouse.cdc.bronze_cdc (
    source_table STRING,
    record_id INT,
    op STRING,
    before_json STRING,
    after_json STRING,
    source_lsn BIGINT,
    source_snapshot STRING,
    ts_ms BIGINT,
    kafka_key STRING,
    raw_value STRING,
    topic STRING,
    kafka_partition INT,
    kafka_offset BIGINT,
    kafka_timestamp TIMESTAMP,
    is_tombstone BOOLEAN,
    bronze_ingested_at TIMESTAMP
) USING iceberg
PARTITIONED BY (source_table)
''')

raw_cdc = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribePattern", "dbserver1\\.public\\.(customers|drivers)")
    .option("startingOffsets", "earliest")
    .load()
)

bronze_cdc_batch = (
    raw_cdc
    .select(
        F.regexp_extract(F.col("topic"), r"dbserver1\.public\.(.+)", 1).alias("source_table"),
        F.get_json_object(F.col("key").cast("string"), "$.payload.id").cast("int").alias("key_id"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.after.id").cast("int").alias("after_id"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.before.id").cast("int").alias("before_id"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.op").alias("op"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.before").alias("before_json"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.after").alias("after_json"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.source.lsn").cast("bigint").alias("source_lsn"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.source.snapshot").alias("source_snapshot"),
        F.get_json_object(F.col("value").cast("string"), "$.payload.ts_ms").cast("bigint").alias("ts_ms"),
        F.col("key").cast("string").alias("kafka_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.col("topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("value").isNull().alias("is_tombstone"),
        F.current_timestamp().alias("bronze_ingested_at")
    )
    .withColumn("record_id", F.coalesce("after_id", "before_id", "key_id"))
    .drop("key_id", "after_id", "before_id")
)

existing_offsets = (
    spark.table("lakehouse.cdc.bronze_cdc")
    .select("topic", "kafka_partition", "kafka_offset")
)

bronze_cdc_new = (
    bronze_cdc_batch.alias("n")
    .join(
        existing_offsets.alias("e"),
        on=["topic", "kafka_partition", "kafka_offset"],
        how="left_anti"
    )
)

new_rows = bronze_cdc_new.count()

if new_rows > 0:
    bronze_cdc_new.writeTo("lakehouse.cdc.bronze_cdc").append()

print("New Bronze rows appended:", new_rows)

spark.sql('''
SELECT source_table, op, is_tombstone, COUNT(*) AS rows
FROM lakehouse.cdc.bronze_cdc
GROUP BY source_table, op, is_tombstone
ORDER BY source_table, op, is_tombstone
''').show(truncate=False)

New Bronze rows appended: 185
+------------+----+------------+----+
|source_table|op  |is_tombstone|rows|
+------------+----+------------+----+
|customers   |NULL|true        |351 |
|customers   |c   |false       |683 |
|customers   |d   |false       |351 |
|customers   |r   |false       |91  |
|customers   |u   |false       |706 |
|drivers     |NULL|true        |227 |
|drivers     |c   |false       |336 |
|drivers     |d   |false       |227 |
|drivers     |r   |false       |15  |
|drivers     |u   |false       |560 |
+------------+----+------------+----+



In [3]:
# Inspect Bronze CDC
spark.sql('''
SELECT
  source_table,
  record_id,
  op,
  is_tombstone,
  source_lsn,
  ts_ms,
  kafka_offset
FROM lakehouse.cdc.bronze_cdc
ORDER BY kafka_timestamp DESC, kafka_offset DESC
LIMIT 30
''').show(truncate=False)

+------------+---------+----+------------+----------+-------------+------------+
|source_table|record_id|op  |is_tombstone|source_lsn|ts_ms        |kafka_offset|
+------------+---------+----+------------+----------+-------------+------------+
|customers   |646      |c   |false       |34518152  |1777404714224|1583        |
|customers   |485      |u   |false       |34517976  |1777404713211|1582        |
|customers   |645      |c   |false       |34517688  |1777404712197|1581        |
|customers   |644      |c   |false       |34517456  |1777404711185|1580        |
|drivers     |284      |c   |false       |34514360  |1777404710173|987         |
|customers   |632      |u   |false       |34514080  |1777404709160|1579        |
|drivers     |246      |u   |false       |34513848  |1777404708147|986         |
|customers   |643      |c   |false       |34513624  |1777404707134|1578        |
|customers   |130      |NULL|true        |NULL      |NULL         |1577        |
|customers   |130      |d   

In [4]:
# Create both Silver tables
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_customers (
    id INT,
    name STRING,
    email STRING,
    country STRING,
    created_at TIMESTAMP,
    last_updated_ms BIGINT
) USING iceberg
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_drivers (
    id INT,
    name STRING,
    license_number STRING,
    rating DOUBLE,
    city STRING,
    active BOOLEAN,
    created_at TIMESTAMP,
    last_updated_ms BIGINT
) USING iceberg
""")

spark.sql("DESCRIBE lakehouse.cdc.silver_customers").show(truncate=False)
spark.sql("DESCRIBE lakehouse.cdc.silver_drivers").show(truncate=False)

+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|id             |int      |NULL   |
|name           |string   |NULL   |
|email          |string   |NULL   |
|country        |string   |NULL   |
|created_at     |timestamp|NULL   |
|last_updated_ms|bigint   |NULL   |
+---------------+---------+-------+

+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|id             |int      |NULL   |
|name           |string   |NULL   |
|license_number |string   |NULL   |
|rating         |double   |NULL   |
|city           |string   |NULL   |
|active         |boolean  |NULL   |
|created_at     |timestamp|NULL   |
|last_updated_ms|bigint   |NULL   |
+---------------+---------+-------+



In [5]:
# Helper for parsing (microseconds issue)
def parse_debezium_ts(json_col, field_name):
    raw = f"get_json_object({json_col}, '$.{field_name}')"
    return F.expr(f"""
        CASE
            WHEN {raw} IS NULL THEN NULL
            WHEN {raw} RLIKE '^[0-9]+$' AND length({raw}) >= 16
                THEN timestamp_micros(CAST({raw} AS BIGINT))
            WHEN {raw} RLIKE '^[0-9]+$' AND length({raw}) >= 13
                THEN timestamp_millis(CAST({raw} AS BIGINT))
            WHEN {raw} RLIKE '^[0-9]+$'
                THEN to_timestamp(from_unixtime(CAST({raw} AS BIGINT)))
            ELSE try_to_timestamp({raw})
        END
    """)

In [36]:
# Customers: latest event per id, then materialise the merge source
customers_latest = (
    spark.table("lakehouse.cdc.bronze_cdc")
    .filter(F.col("source_table") == "customers")
    .filter(~F.col("is_tombstone"))
    .filter(F.col("record_id").isNotNull())
    .filter(F.col("op").isNotNull())
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("record_id").orderBy(
                F.col("ts_ms").desc(),
                F.col("kafka_offset").desc()
            )
        )
    )
    .filter("rn = 1")
    .select(
        F.col("record_id").alias("id"),
        "op",
        F.get_json_object("after_json", "$.name").alias("name"),
        F.get_json_object("after_json", "$.email").alias("email"),
        F.get_json_object("after_json", "$.country").alias("country"),
        parse_debezium_ts("after_json", "created_at").alias("created_at"),
        F.col("ts_ms").alias("last_updated_ms")
    )
)

customers_latest.show(20, truncate=False)
customers_latest.printSchema()

customer_rows = customers_latest.collect()
customers_stage = spark.createDataFrame(customer_rows, customers_latest.schema)
customers_stage.createOrReplaceTempView("customers_cdc_latest_stage")

+---+---+----------------+-------------------------+-------+--------------------------+---------------+
|id |op |name            |email                    |country|created_at                |last_updated_ms|
+---+---+----------------+-------------------------+-------+--------------------------+---------------+
|148|d  |NULL            |NULL                     |NULL   |NULL                      |1777403586466  |
|463|d  |NULL            |NULL                     |NULL   |NULL                      |1777404217625  |
|471|d  |NULL            |NULL                     |NULL   |NULL                      |1777405259071  |
|496|d  |NULL            |NULL                     |NULL   |NULL                      |1777404995516  |
|243|u  |Alex Kim        |alex.kim@example.com     |Finland|2026-04-28 19:01:45.627992|1777405513686  |
|392|c  |Olivia Garcia   |olivia.garcia@mail.com   |Estonia|2026-04-28 19:13:43.609673|1777403624023  |
|540|u  |Alex Smith      |alex.smith@mail.com      |Estonia|2026

In [37]:
# Customers MERGE
spark.sql('''
MERGE INTO lakehouse.cdc.silver_customers AS t
USING customers_cdc_latest_stage AS s
ON t.id = s.id

WHEN MATCHED AND s.op = 'd' THEN DELETE

WHEN MATCHED AND s.op IN ('c', 'u', 'r') THEN UPDATE SET
  t.name = s.name,
  t.email = s.email,
  t.country = s.country,
  t.created_at = s.created_at,
  t.last_updated_ms = s.last_updated_ms

WHEN NOT MATCHED AND s.op IN ('c', 'u', 'r') THEN INSERT (
  id, name, email, country, created_at, last_updated_ms
) VALUES (
  s.id, s.name, s.email, s.country, s.created_at, s.last_updated_ms
)
''')

spark.sql("SELECT COUNT(*) AS silver_customer_rows FROM lakehouse.cdc.silver_customers").show()

spark.sql('''
SELECT *
FROM lakehouse.cdc.silver_customers
ORDER BY id
LIMIT 20
''').show(truncate=False)

+--------------------+
|silver_customer_rows|
+--------------------+
|                 423|
+--------------------+

+---+---------------+---------------------------+---------+--------------------------+---------------+
|id |name           |email                      |country  |created_at                |last_updated_ms|
+---+---------------+---------------------------+---------+--------------------------+---------------+
|4  |David Jonaitis |david@example.com          |Latvia   |2026-04-28 18:45:23.100516|1777404951765  |
|11 |Lars Johansson |updated_11_677@mail.com    |Poland   |2026-04-28 18:46:30.459349|1777403099039  |
|17 |Zara Mets      |updated_17_147@test.net    |Norway   |2026-04-28 18:46:51.855617|1777402841734  |
|30 |Yuki Johansson |updated_30_269@mail.com    |Estonia  |2026-04-28 18:47:28.866786|1777402582159  |
|34 |Alex Nakamura  |updated_34_358@example.com |France   |2026-04-28 18:47:47.288038|1777404683197  |
|36 |Diego Ozols    |updated_36_414@mail.com    |Sweden   |2

In [8]:
# Helper to to decode Debezium base64 decimals

import base64
from decimal import Decimal
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

def decode_debezium_decimal(value, scale=2):
    if value is None:
        return None
    try:
        # Already numeric-looking
        return float(value)
    except:
        pass

    try:
        raw = base64.b64decode(value)
        integer_value = int.from_bytes(raw, byteorder="big", signed=True)
        return float(Decimal(integer_value) / (Decimal(10) ** scale))
    except:
        return None

decode_driver_rating = udf(lambda x: decode_debezium_decimal(x, 2), DoubleType())

In [38]:
# Drivers: latest event per id, then materialise the merge source
drivers_latest = (
    spark.table("lakehouse.cdc.bronze_cdc")
    .filter(F.col("source_table") == "drivers")
    .filter(~F.col("is_tombstone"))
    .filter(F.col("record_id").isNotNull())
    .filter(F.col("op").isNotNull())
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("record_id").orderBy(
                F.col("ts_ms").desc(),
                F.col("kafka_offset").desc()
            )
        )
    )
    .filter("rn = 1")
    .select(
        F.col("record_id").alias("id"),
        "op",
        F.get_json_object("after_json", "$.name").alias("name"),
        F.get_json_object("after_json", "$.license_number").alias("license_number"),
        decode_driver_rating(F.get_json_object("after_json", "$.rating")).alias("rating"),
        F.get_json_object("after_json", "$.city").alias("city"),
        F.get_json_object("after_json", "$.active").cast("boolean").alias("active"),
        parse_debezium_ts("after_json", "created_at").alias("created_at"),
        F.col("ts_ms").alias("last_updated_ms")
    )
)

drivers_latest.show(20, truncate=False)
drivers_latest.printSchema()

driver_rows = drivers_latest.collect()
drivers_stage = spark.createDataFrame(driver_rows, drivers_latest.schema)
drivers_stage.createOrReplaceTempView("drivers_cdc_latest_stage")

+---+---+--------------+--------------+------+-------------+------+--------------------------+---------------+
|id |op |name          |license_number|rating|city         |active|created_at                |last_updated_ms|
+---+---+--------------+--------------+------+-------------+------+--------------------------+---------------+
|148|d  |NULL          |NULL          |NULL  |NULL         |NULL  |NULL                      |1777404204945  |
|243|d  |NULL          |NULL          |NULL  |NULL         |NULL  |NULL                      |1777404577987  |
|85 |d  |NULL          |NULL          |NULL  |NULL         |NULL  |NULL                      |1777402865601  |
|137|d  |NULL          |NULL          |NULL  |NULL         |NULL  |NULL                      |1777403480811  |
|251|u  |Fatima Mets   |TLC-20445     |3.68  |Staten Island|false |2026-04-28 19:25:42.976573|1777404958865  |
|65 |d  |NULL          |NULL          |NULL  |NULL         |NULL  |NULL                      |1777402687262  |
|

In [39]:
# Drivers MERGE
spark.sql('''
MERGE INTO lakehouse.cdc.silver_drivers AS t
USING drivers_cdc_latest_stage AS s
ON t.id = s.id

WHEN MATCHED AND s.op = 'd' THEN DELETE

WHEN MATCHED AND s.op IN ('c', 'u', 'r') THEN UPDATE SET
  t.name = s.name,
  t.license_number = s.license_number,
  t.rating = s.rating,
  t.city = s.city,
  t.active = s.active,
  t.created_at = s.created_at,
  t.last_updated_ms = s.last_updated_ms

WHEN NOT MATCHED AND s.op IN ('c', 'u', 'r') THEN INSERT (
  id, name, license_number, rating, city, active, created_at, last_updated_ms
) VALUES (
  s.id, s.name, s.license_number, s.rating, s.city, s.active, s.created_at, s.last_updated_ms
)
''')

spark.sql("SELECT COUNT(*) AS silver_driver_rows FROM lakehouse.cdc.silver_drivers").show()

spark.sql('''
SELECT *
FROM lakehouse.cdc.silver_drivers
ORDER BY id
LIMIT 20
''').show(truncate=False)

+------------------+
|silver_driver_rows|
+------------------+
|               124|
+------------------+

+---+-------------+--------------+------+-------------+------+--------------------------+---------------+
|id |name         |license_number|rating|city         |active|created_at                |last_updated_ms|
+---+-------------+--------------+------+-------------+------+--------------------------+---------------+
|105|Lena Garcia  |TLC-59941     |4.91  |Bronx        |false |2026-04-28 19:02:47.499879|1777405449526  |
|118|Amir Nakamura|TLC-91594     |4.15  |Staten Island|true  |2026-04-28 19:05:27.643141|1777404563666  |
|134|Kai Ozols    |TLC-80671     |3.63  |Staten Island|false |2026-04-28 19:08:12.769464|1777405012352  |
|136|Zara Mets    |TLC-28559     |4.07  |Staten Island|true  |2026-04-28 19:08:17.953004|1777405005163  |
|138|Diego Larsen |TLC-99120     |3.86  |Brooklyn     |false |2026-04-28 19:08:27.313368|1777405115618  |
|144|Emma Kowalski|TLC-63028     |3.87  |State

In [41]:
# Validate Silver against PostgreSQL: counts + spot-check rows
import psycopg2

conn = psycopg2.connect(
    host=os.environ.get("PG_HOST", "postgres"),
    port=int(os.environ.get("PG_PORT", 5432)),
    dbname=os.environ.get("PG_DB", "sourcedb"),
    user=os.environ.get("PG_USER", "cdc_user"),
    password=os.environ.get("PG_PASSWORD", "cdc_pass"),
)
conn.autocommit = True
cur = conn.cursor()

for source_table, silver_table in [
    ("customers", "lakehouse.cdc.silver_customers"),
    ("drivers", "lakehouse.cdc.silver_drivers"),
]:
    cur.execute(f"SELECT COUNT(*) FROM {source_table}")
    pg_count = cur.fetchone()[0]
    silver_count = spark.table(silver_table).count()
    print(f"{source_table}: postgres={pg_count}, silver={silver_count}")

    print(f"\nPostgreSQL sample: {source_table}")
    cur.execute(f"SELECT * FROM {source_table} ORDER BY id LIMIT 3")
    for row in cur.fetchall():
        print(row)

    print(f"\nIceberg sample: {silver_table}")
    spark.table(silver_table).orderBy("id").show(3, truncate=False)
    print("-" * 80)

cur.close()
conn.close()

customers: postgres=433, silver=423

PostgreSQL sample: customers
(4, 'David Jonaitis', 'david@example.com', 'Latvia', datetime.datetime(2026, 4, 28, 18, 45, 23, 100516))
(11, 'Lars Johansson', 'updated_11_677@mail.com', 'Poland', datetime.datetime(2026, 4, 28, 18, 46, 30, 459349))
(30, 'Yuki Johansson', 'updated_30_269@mail.com', 'Estonia', datetime.datetime(2026, 4, 28, 18, 47, 28, 866786))

Iceberg sample: lakehouse.cdc.silver_customers
+---+--------------+-----------------------+-------+--------------------------+---------------+
|id |name          |email                  |country|created_at                |last_updated_ms|
+---+--------------+-----------------------+-------+--------------------------+---------------+
|4  |David Jonaitis|david@example.com      |Latvia |2026-04-28 18:45:23.100516|1777404951765  |
|11 |Lars Johansson|updated_11_677@mail.com|Poland |2026-04-28 18:46:30.459349|1777403099039  |
|17 |Zara Mets     |updated_17_147@test.net|Norway |2026-04-28 18:46:51.8556

In [12]:
# Snapshot history for the report
spark.sql('''
SELECT *
FROM lakehouse.cdc.silver_customers.history
ORDER BY made_current_at DESC
''').show(truncate=False)

spark.sql('''
SELECT *
FROM lakehouse.cdc.silver_drivers.history
ORDER BY made_current_at DESC
''').show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-04-28 19:32:40.708|4048394606627733340|3289416714107596531|true               |
|2026-04-28 18:56:47.618|3289416714107596531|NULL               |true               |
+-----------------------+-------------------+-------------------+-------------------+

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-04-28 19:33:16.269|3163813203510872929|2011039556538289646|true               |
|2026-04-28 18:57:15.333|2011039556538289646|NULL               |true               |
+-----------------------+-------------------+--------

In [13]:
# Taxi Bronze: raw Kafka JSON + Kafka metadata, append-only, idempotent

spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.taxi")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze_taxi (
    kafka_key STRING,
    raw_json STRING,
    topic STRING,
    kafka_partition INT,
    kafka_offset BIGINT,
    kafka_timestamp TIMESTAMP,
    bronze_ingested_at TIMESTAMP
) USING iceberg
PARTITIONED BY (days(kafka_timestamp))
""")

DataFrame[]

In [14]:
# Batch load from Kafka topic taxi-trips
# Safe to rerun: only unseen (topic, partition, offset) rows are appended

raw_taxi = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "taxi-trips")
    .option("startingOffsets", "earliest")
    .load()
)

bronze_taxi_batch = (
    raw_taxi
    .select(
        F.col("key").cast("string").alias("kafka_key"),
        F.col("value").cast("string").alias("raw_json"),
        F.col("topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.current_timestamp().alias("bronze_ingested_at")
    )
)

existing_taxi_offsets = (
    spark.table("lakehouse.taxi.bronze_taxi")
    .select("topic", "kafka_partition", "kafka_offset")
)

bronze_taxi_new = (
    bronze_taxi_batch.alias("n")
    .join(
        existing_taxi_offsets.alias("e"),
        on=["topic", "kafka_partition", "kafka_offset"],
        how="left_anti"
    )
)

new_rows = bronze_taxi_new.count()
if new_rows > 0:
    bronze_taxi_new.writeTo("lakehouse.taxi.bronze_taxi").append()

print("New Bronze taxi rows appended:", new_rows)

spark.sql("""
SELECT COUNT(*) AS bronze_taxi_rows
FROM lakehouse.taxi.bronze_taxi
""").show()

New Bronze taxi rows appended: 0
+----------------+
|bronze_taxi_rows|
+----------------+
|            3054|
+----------------+



In [15]:
# Taxi schema and zone lookup
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
])

zones = spark.read.parquet("/home/jovyan/project/data/taxi_zone_lookup.parquet")

pickup_zones = zones.select(
    F.col("LocationID").alias("pu_location_id"),
    F.col("Zone").alias("pickup_zone"),
    F.col("Borough").alias("pickup_borough")
)

dropoff_zones = zones.select(
    F.col("LocationID").alias("do_location_id"),
    F.col("Zone").alias("dropoff_zone"),
    F.col("Borough").alias("dropoff_borough")
)

In [16]:
# Taxi Silver table
# Taxi Silver table (create once, do not drop on reruns)
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.silver_taxi (
    trip_id STRING,
    vendor_id INT,
    pickup_ts TIMESTAMP,
    dropoff_ts TIMESTAMP,
    pickup_date DATE,
    pickup_hour INT,
    trip_duration_minutes DOUBLE,
    passenger_count INT,
    trip_distance DOUBLE,
    pu_location_id INT,
    pickup_zone STRING,
    pickup_borough STRING,
    do_location_id INT,
    dropoff_zone STRING,
    dropoff_borough STRING,
    fare_amount DOUBLE,
    tip_amount DOUBLE,
    total_amount DOUBLE,
    payment_type INT,
    congestion_surcharge DOUBLE,
    raw_json STRING
) USING iceberg
PARTITIONED BY (pickup_date)
""")

DataFrame[]

In [17]:
# Transform Bronze taxi into Silver, then MERGE from a staged table

taxi_parsed = (
    spark.table("lakehouse.taxi.bronze_taxi")
    .select(
        F.from_json("raw_json", taxi_schema).alias("j"),
        "raw_json",
        "kafka_timestamp",
        "kafka_offset"
    )
    .select("j.*", "raw_json", "kafka_timestamp", "kafka_offset")
)

taxi_silver_source = (
    taxi_parsed
    .withColumn("pickup_ts", F.to_timestamp("tpep_pickup_datetime"))
    .withColumn("dropoff_ts", F.to_timestamp("tpep_dropoff_datetime"))
    .withColumn("pickup_date", F.to_date("pickup_ts"))
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
    .withColumn("pu_location_id", F.col("PULocationID").cast("int"))
    .withColumn("do_location_id", F.col("DOLocationID").cast("int"))
    .withColumn(
        "trip_duration_minutes",
        (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60.0
    )
    .withColumn(
        "trip_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("VendorID").cast("string"), F.lit("")),
                F.coalesce(F.col("tpep_pickup_datetime"), F.lit("")),
                F.coalesce(F.col("tpep_dropoff_datetime"), F.lit("")),
                F.coalesce(F.col("PULocationID").cast("string"), F.lit("")),
                F.coalesce(F.col("DOLocationID").cast("string"), F.lit("")),
                F.coalesce(F.col("trip_distance").cast("string"), F.lit("")),
                F.coalesce(F.col("fare_amount").cast("string"), F.lit("")),
                F.coalesce(F.col("tip_amount").cast("string"), F.lit("")),
                F.coalesce(F.col("total_amount").cast("string"), F.lit(""))
            ),
            256
        )
    )
    .filter(F.col("pickup_ts").isNotNull())
    .filter(F.col("dropoff_ts").isNotNull())
    .filter(F.col("pu_location_id").isNotNull())
    .filter(F.col("do_location_id").isNotNull())
    .filter(F.col("dropoff_ts") >= F.col("pickup_ts"))
    .filter(F.col("trip_duration_minutes") > 0)
    .filter(F.col("trip_duration_minutes") <= 720)
    .filter(F.col("trip_distance") >= 0)
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("tip_amount") >= 0)
    .filter(F.col("total_amount") >= 0)
    .join(pickup_zones, on="pu_location_id", how="left")
    .join(dropoff_zones, on="do_location_id", how="left")
    .select(
        "trip_id",
        F.col("VendorID").alias("vendor_id"),
        "pickup_ts",
        "dropoff_ts",
        "pickup_date",
        "pickup_hour",
        "trip_duration_minutes",
        "passenger_count",
        "trip_distance",
        "pu_location_id",
        "pickup_zone",
        "pickup_borough",
        "do_location_id",
        "dropoff_zone",
        "dropoff_borough",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type",
        "congestion_surcharge",
        "raw_json",
        "kafka_timestamp",
        "kafka_offset"
    )
)

w = Window.partitionBy("trip_id").orderBy(
    F.col("kafka_timestamp").desc(),
    F.col("kafka_offset").desc()
)

taxi_silver_latest = (
    taxi_silver_source
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn", "kafka_timestamp", "kafka_offset")
)

# Scratch stage table, fine to recreate
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.silver_taxi_stage")

taxi_silver_latest.writeTo("lakehouse.taxi.silver_taxi_stage").using("iceberg").create()

spark.sql("""
MERGE INTO lakehouse.taxi.silver_taxi AS t
USING lakehouse.taxi.silver_taxi_stage AS s
ON t.trip_id = s.trip_id

WHEN MATCHED THEN UPDATE SET
  t.vendor_id = s.vendor_id,
  t.pickup_ts = s.pickup_ts,
  t.dropoff_ts = s.dropoff_ts,
  t.pickup_date = s.pickup_date,
  t.pickup_hour = s.pickup_hour,
  t.trip_duration_minutes = s.trip_duration_minutes,
  t.passenger_count = s.passenger_count,
  t.trip_distance = s.trip_distance,
  t.pu_location_id = s.pu_location_id,
  t.pickup_zone = s.pickup_zone,
  t.pickup_borough = s.pickup_borough,
  t.do_location_id = s.do_location_id,
  t.dropoff_zone = s.dropoff_zone,
  t.dropoff_borough = s.dropoff_borough,
  t.fare_amount = s.fare_amount,
  t.tip_amount = s.tip_amount,
  t.total_amount = s.total_amount,
  t.payment_type = s.payment_type,
  t.congestion_surcharge = s.congestion_surcharge,
  t.raw_json = s.raw_json

WHEN NOT MATCHED THEN INSERT (
  trip_id, vendor_id, pickup_ts, dropoff_ts, pickup_date, pickup_hour,
  trip_duration_minutes, passenger_count, trip_distance, pu_location_id,
  pickup_zone, pickup_borough, do_location_id, dropoff_zone,
  dropoff_borough, fare_amount, tip_amount, total_amount,
  payment_type, congestion_surcharge, raw_json
) VALUES (
  s.trip_id, s.vendor_id, s.pickup_ts, s.dropoff_ts, s.pickup_date, s.pickup_hour,
  s.trip_duration_minutes, s.passenger_count, s.trip_distance, s.pu_location_id,
  s.pickup_zone, s.pickup_borough, s.do_location_id, s.dropoff_zone,
  s.dropoff_borough, s.fare_amount, s.tip_amount, s.total_amount,
  s.payment_type, s.congestion_surcharge, s.raw_json
)
""")

spark.sql("""
SELECT COUNT(*) AS silver_taxi_rows
FROM lakehouse.taxi.silver_taxi
""").show()

spark.sql("""
SELECT *
FROM lakehouse.taxi.silver_taxi
LIMIT 10
""").show(truncate=False)

+----------------+
|silver_taxi_rows|
+----------------+
|            2997|
+----------------+

+----------------------------------------------------------------+---------+-------------------+-------------------+-----------+-----------+---------------------+---------------+-------------+--------------+-----------------------------+--------------+--------------+-----------------------------+---------------+-----------+----------+------------+------------+--------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|trip_id                          

In [18]:
# Build Gold from Silver without duplicates

gold_taxi = (
    spark.table("lakehouse.taxi.silver_taxi")
    .filter(F.col("pickup_date").isNotNull())
    .filter(F.col("pickup_hour").isNotNull())
    .filter(F.col("pickup_zone").isNotNull())
    .groupBy("pickup_date", "pickup_hour", "pickup_zone")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
    )
)

print("Gold source row count:", gold_taxi.count())

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold_taxi_hourly_zone (
    pickup_date DATE,
    pickup_hour INT,
    pickup_zone STRING,
    trip_count BIGINT,
    avg_fare_amount DOUBLE,
    avg_total_amount DOUBLE,
    avg_trip_distance DOUBLE,
    avg_trip_duration_minutes DOUBLE
) USING iceberg
PARTITIONED BY (pickup_date)
""")

gold_taxi.writeTo("lakehouse.taxi.gold_taxi_hourly_zone").overwritePartitions()

spark.sql("""
SELECT *
FROM lakehouse.taxi.gold_taxi_hourly_zone
ORDER BY pickup_date DESC, pickup_hour DESC, trip_count DESC
LIMIT 20
""").show(truncate=False)

Gold source row count: 93
+-----------+-----------+-----------------------------+----------+---------------+----------------+-----------------+-------------------------+
|pickup_date|pickup_hour|pickup_zone                  |trip_count|avg_fare_amount|avg_total_amount|avg_trip_distance|avg_trip_duration_minutes|
+-----------+-----------+-----------------------------+----------+---------------+----------------+-----------------+-------------------------+
|2025-01-01 |1          |N/A                          |1         |14.2           |23.04           |2.17             |12.75                    |
|2025-01-01 |1          |World Trade Center           |1         |32.4           |44.88           |6.78             |25.73                    |
|2025-01-01 |1          |Lincoln Square East          |1         |12.1           |20.52           |2.04             |10.98                    |
|2025-01-01 |1          |Midtown North                |1         |13.5           |22.2            |2.03       

In [19]:
# Outputs for report
print("CDC counts")
spark.sql("SELECT COUNT(*) FROM lakehouse.cdc.bronze_cdc").show()
spark.sql("SELECT COUNT(*) FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT COUNT(*) FROM lakehouse.cdc.silver_drivers").show()

print("Taxi counts")
spark.sql("SELECT COUNT(*) FROM lakehouse.taxi.bronze_taxi").show()
spark.sql("SELECT COUNT(*) FROM lakehouse.taxi.silver_taxi").show()
spark.sql("SELECT COUNT(*) FROM lakehouse.taxi.gold_taxi_hourly_zone").show()

CDC counts
+--------+
|count(1)|
+--------+
|    2572|
+--------+

+--------+
|count(1)|
+--------+
|     349|
+--------+

+--------+
|count(1)|
+--------+
|      78|
+--------+

Taxi counts
+--------+
|count(1)|
+--------+
|    3054|
+--------+

+--------+
|count(1)|
+--------+
|    2997|
+--------+

+--------+
|count(1)|
+--------+
|      93|
+--------+



In [20]:
# Iceberg snapshot history for report screenshots
spark.sql("""
SELECT *
FROM lakehouse.cdc.silver_customers.history
ORDER BY made_current_at DESC
""").show(truncate=False)

spark.sql("""
SELECT *
FROM lakehouse.taxi.silver_taxi.history
ORDER BY made_current_at DESC
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-04-28 19:32:40.708|4048394606627733340|3289416714107596531|true               |
|2026-04-28 18:56:47.618|3289416714107596531|NULL               |true               |
+-----------------------+-------------------+-------------------+-------------------+

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-04-28 19:34:26.956|1322701720260801721|4130140958933561406|true               |
|2026-04-28 19:12:09.984|4130140958933561406|NULL               |true               |
+-----------------------+-------------------+--------